# Phase 7B - F4 Role-Aware Measurement Availability Interaction Probe

Experiment ID: `phase7b_role_availability_interaction_v1`

Run ID: `phase7b_f4_single_probe_20260612`

This notebook evaluates exactly one deferred F4 hypothesis:

`F4 = F2 + available_measurement_count x Player_Type`

It compares F4 against persisted Phase 7 F2 OOF predictions. It does not retrain
F2, generate submissions, run HPO, compare model families, use `School` as a
feature, use leaderboard data, or open Phase 8.

## 1. Imports, Constants, and Scope Gates

**Purpose.** Pin the authorized run, paths, features, thresholds, and scope.

**Expected output.** Constants and artifact paths for a fail-closed run.

In [1]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "data" / "input" / "train.csv").exists() and (candidate / ".git").exists():
            return candidate
    raise RuntimeError(f"Could not locate project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
EXPERIMENT_ID = "phase7b_role_availability_interaction_v1"
RUN_ID = "phase7b_f4_single_probe_20260612"
VARIANT_ID = "phase7b_f4_role_count_player_type"
PROJECT_SEED = 42
RARE_SCHOOL_THRESHOLD = 5
OOF_GAIN_THRESHOLD = 0.005436
SAME_SIGN_FOLD_THRESHOLD = 4
MIN_SLICE_SIZE = 50
SLICE_DEGRADATION_THRESHOLD = 0.02
EXPECTED_F2_AUC = 0.8116502602456482

DATA_DIR = PROJECT_ROOT / "data" / "input"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
FOLDS_PATH = OUTPUTS_DIR / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"
F2_OOF_PATH = OUTPUTS_DIR / "oof" / "phase7_missingness_availability_v1_phase7_f2_median_flags_count_oof_predictions.csv"
MAIN_LOG_PATH = PROJECT_ROOT / "logs" / "experiment_log.csv"
PHASE7_ACCEPTANCE_PATH = PROJECT_ROOT / "docs" / "07_feature_engineering" / "phase7_acceptance.md"

OOF_PATH = OUTPUTS_DIR / "oof" / f"{EXPERIMENT_ID}_{VARIANT_ID}_oof_predictions.csv"
VARIANT_SUMMARY_PATH = OUTPUTS_DIR / "validation" / f"{EXPERIMENT_ID}_variant_summary.csv"
SLICE_REPORT_PATH = OUTPUTS_DIR / "validation" / f"{EXPERIMENT_ID}_slice_report.csv"
VALIDATION_REPORT_PATH = OUTPUTS_DIR / "reports" / f"{EXPERIMENT_ID}_validation_report.md"
CANDIDATE_LOG_PATH = OUTPUTS_DIR / "reports" / f"{EXPERIMENT_ID}_experiment_log_candidate.csv"

BASE_FEATURES = [
    "Year",
    "Age",
    "Height",
    "Weight",
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
    "Player_Type",
    "Position_Type",
    "Position",
]
CATEGORICAL_FEATURES = ["Player_Type", "Position_Type", "Position"]
PHYSICAL_TEST_COLUMNS = [
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
]
MISSINGNESS_FLAGS = [
    "Age_missing",
    "Sprint_40yd_missing",
    "Vertical_Jump_missing",
    "Bench_Press_Reps_missing",
    "Broad_Jump_missing",
    "Agility_3cone_missing",
    "Shuttle_missing",
]
F2_FEATURES = BASE_FEATURES + MISSINGNESS_FLAGS + ["available_measurement_count"]
INTERACTION_SOURCE_COLUMNS = ["Player_Type", "available_measurement_count"]
FORBIDDEN_FEATURES = {"Id", "Drafted", "School", "measurement_completeness_group", "frequent_vs_rare_school_group"}

ARTIFACT_PATHS = [
    OOF_PATH,
    VARIANT_SUMMARY_PATH,
    SLICE_REPORT_PATH,
    VALIDATION_REPORT_PATH,
    CANDIDATE_LOG_PATH,
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}; pandas: {pd.__version__}; sklearn: {sklearn.__version__}; numpy: {np.__version__}")

Project root: C:\GitHub\reto_Tokio
Python: 3.13.13; pandas: 3.0.3; sklearn: 1.9.0; numpy: 2.4.6


## 2. Git, Artifact, and Path Gates

**Purpose.** Stop before training if HEAD, staged files, forbidden diffs, logs, or
artifact collisions violate the authorization.

**Expected output.** Gate summary.

In [2]:
def run_git(args: list[str]) -> str:
    result = subprocess.run(["git", *args], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True)
    return result.stdout.strip()


def sha256_16(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()[:16]


def assert_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Required {label} missing: {path}")


def assert_not_exists(path: Path, label: str) -> None:
    if path.exists():
        raise FileExistsError(f"Artifact collision for {label}: {path}")


def read_bytes(path: Path) -> bytes:
    return path.read_bytes() if path.exists() else b"__MISSING__"


head = run_git(["rev-parse", "--short", "HEAD"])
status_short = run_git(["status", "--short"])
staged_files = run_git(["diff", "--cached", "--name-only"])
diff_check = subprocess.run(["git", "diff", "--check"], cwd=PROJECT_ROOT, capture_output=True, text=True)
forbidden_diff = run_git([
    "diff",
    "--name-only",
    "--",
    "data/input",
    "notebooks/_official",
    "references",
    "outputs/submissions",
    "logs/experiment_log.csv",
    ".vscode/settings.json",
])

if head != "8b21db5":
    raise RuntimeError(f"Authorized HEAD mismatch: expected 8b21db5 or documented successor, got {head}")
if staged_files:
    raise RuntimeError(f"Staged files exist before Phase 7B execution: {staged_files}")
if diff_check.returncode != 0 or diff_check.stdout.strip() or diff_check.stderr.strip():
    raise RuntimeError(f"git diff --check failed: {diff_check.stdout}{diff_check.stderr}")
if forbidden_diff:
    raise RuntimeError(f"Forbidden path diff before execution: {forbidden_diff}")

for path, label in [
    (TRAIN_PATH, "train.csv"),
    (TEST_PATH, "test.csv"),
    (FOLDS_PATH, "frozen fold assignments"),
    (F2_OOF_PATH, "Phase 7 F2 OOF"),
    (MAIN_LOG_PATH, "main experiment log"),
    (PHASE7_ACCEPTANCE_PATH, "Phase 7 acceptance draft"),
]:
    assert_exists(path, label)

acceptance_text = PHASE7_ACCEPTANCE_PATH.read_text(encoding="utf-8")
if "ACCEPT WITH WARNINGS" not in acceptance_text or "phase7_f2_median_flags_count" not in acceptance_text:
    raise RuntimeError("Phase 7 acceptance draft does not report ACCEPT WITH WARNINGS for F2.")

for path in ARTIFACT_PATHS:
    assert_not_exists(path, path.name)

main_log_before = read_bytes(MAIN_LOG_PATH)

gate_summary = {
    "head": head,
    "staged_files": staged_files,
    "forbidden_diff_clean": forbidden_diff == "",
    "artifact_collisions": 0,
    "phase8_locked": True,
}
gate_summary

{'head': '8b21db5',
 'staged_files': '',
 'forbidden_diff_clean': True,
 'artifact_collisions': 0,
 'phase8_locked': True}

## 3. Data, Folds, and F2 Baseline Reference

**Purpose.** Load official train/test only for contract checks, verify frozen
folds, and validate persisted F2 OOF. F2 is not retrained.

**Expected output.** Fold array, target, and F2 metrics.

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
folds = pd.read_csv(FOLDS_PATH)
f2_oof = pd.read_csv(F2_OOF_PATH)

if train.shape != (2781, 16):
    raise RuntimeError(f"Unexpected train shape: {train.shape}")
if test.shape[0] != 696:
    raise RuntimeError(f"Unexpected test row count: {test.shape[0]}")
if "Drafted" not in train.columns or "Drafted" in test.columns:
    raise RuntimeError("Target contract failed.")
if not {"Id", "Drafted", "School"}.issubset(train.columns):
    raise RuntimeError("Required train columns missing.")
if sorted(train["Drafted"].dropna().unique().tolist()) != [0, 1]:
    raise RuntimeError("Target labels are not exactly 0 and 1.")

fold_hash = sha256_16(FOLDS_PATH)
if fold_hash != "96937649526bcadb":
    raise RuntimeError(f"Frozen fold hash mismatch: {fold_hash}")
if folds.shape != (2781, 2) or list(folds.columns) != ["Id", "fold"]:
    raise RuntimeError(f"Unexpected fold file shape/columns: {folds.shape}, {list(folds.columns)}")
if sorted(folds["fold"].unique().tolist()) != [0, 1, 2, 3, 4]:
    raise RuntimeError("Frozen fold labels are not exactly 0..4.")
if not folds["Id"].equals(train["Id"]):
    raise RuntimeError("Frozen fold Id order does not match train Id order.")

required_oof_cols = ["Id", "fold", "y_true", "y_pred_proba"]
if list(f2_oof.columns) != required_oof_cols:
    raise RuntimeError(f"F2 OOF columns mismatch: {list(f2_oof.columns)}")
if len(f2_oof) != 2781:
    raise RuntimeError(f"F2 OOF row count mismatch: {len(f2_oof)}")
if not f2_oof["Id"].equals(train["Id"]) or not f2_oof["fold"].equals(folds["fold"]):
    raise RuntimeError("F2 OOF Id/fold alignment mismatch.")
if not np.array_equal(f2_oof["y_true"].astype(int).to_numpy(), train["Drafted"].astype(int).to_numpy()):
    raise RuntimeError("F2 OOF y_true mismatch.")
if not np.isfinite(f2_oof["y_pred_proba"]).all() or f2_oof["y_pred_proba"].isna().any() or not f2_oof["y_pred_proba"].between(0, 1).all():
    raise RuntimeError("F2 OOF probabilities invalid.")

fold_array = folds["fold"].to_numpy()
y = train["Drafted"].astype(int).to_numpy()
f2_proba = f2_oof["y_pred_proba"].to_numpy()
f2_oof_auc = float(roc_auc_score(y, f2_proba))
if abs(f2_oof_auc - EXPECTED_F2_AUC) > 1e-12:
    raise RuntimeError(f"F2 OOF AUC mismatch: expected {EXPECTED_F2_AUC}, got {f2_oof_auc}")

for fold_id in range(5):
    if len(np.unique(y[fold_array == fold_id])) != 2:
        raise RuntimeError(f"Fold {fold_id} is one-class.")

contract_summary = {
    "train_shape": train.shape,
    "test_shape": test.shape,
    "fold_hash": fold_hash,
    "f2_oof_auc": f2_oof_auc,
}
contract_summary

{'train_shape': (2781, 16),
 'test_shape': (696, 15),
 'fold_hash': '96937649526bcadb',
 'f2_oof_auc': 0.8116502602456482}

## 4. Row-Wise F2 Feature Builders

**Purpose.** Recreate the accepted F2 feature matrix using only row-wise,
parameter-free missingness and availability features.

**Expected output.** `train_fe` with F2 features and diagnostic slice fields.

In [4]:
def add_f2_rowwise_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Age_missing"] = out["Age"].isna().astype(int)
    for column in PHYSICAL_TEST_COLUMNS:
        out[f"{column}_missing"] = out[column].isna().astype(int)
    out["available_measurement_count"] = out[PHYSICAL_TEST_COLUMNS].notna().sum(axis=1).astype(int)
    count = out["available_measurement_count"]
    out["measurement_completeness_group"] = np.select(
        [count.eq(0), count.between(1, 3), count.between(4, 5), count.eq(6)],
        [0, 1, 2, 3],
        default=-1,
    ).astype(int)
    if (out["measurement_completeness_group"] < 0).any():
        raise RuntimeError("Invalid measurement_completeness_group value.")
    return out


for column in F2_FEATURES:
    source = column.replace("_missing", "") if column.endswith("_missing") else column
    if column not in MISSINGNESS_FLAGS and column != "available_measurement_count" and source not in train.columns:
        raise RuntimeError(f"Required source column missing: {source}")

train_fe = add_f2_rowwise_features(train)
school_key = train_fe["School"].astype("string").fillna("__MISSING__")
school_counts = school_key.value_counts(dropna=False)
train_fe["frequent_vs_rare_school_group"] = np.where(
    school_key.map(school_counts).to_numpy() < RARE_SCHOOL_THRESHOLD,
    "rare",
    "frequent",
)

feature_overlap = sorted(FORBIDDEN_FEATURES.intersection(F2_FEATURES))
if feature_overlap:
    raise RuntimeError(f"Forbidden F2 feature present: {feature_overlap}")
if not train_fe["available_measurement_count"].between(0, 6).all():
    raise RuntimeError("available_measurement_count outside 0..6")

feature_summary = {
    "f2_feature_count": len(F2_FEATURES),
    "available_measurement_count_counts": train_fe["available_measurement_count"].value_counts().sort_index().to_dict(),
    "school_group_counts_diagnostic_only": train_fe["frequent_vs_rare_school_group"].value_counts().to_dict(),
}
feature_summary

{'f2_feature_count': 21,
 'available_measurement_count_counts': {0: 56,
  1: 240,
  2: 236,
  3: 137,
  4: 269,
  5: 454,
  6: 1389},
 'school_group_counts_diagnostic_only': {'frequent': 2559, 'rare': 222}}

## 5. Fold-Safe F4 Pipeline

**Purpose.** Add exactly one interaction family:
`available_measurement_count x Player_Type`. The `Player_Type` encoder is fit
inside each training fold.

**Expected output.** Training helpers for exactly one F4 variant.

In [5]:
class PlayerAvailabilityInteraction(BaseEstimator, TransformerMixin):
    """Fold-fitted Player_Type OHE multiplied by row-wise availability count."""

    def __init__(self, player_col: str = "Player_Type", count_col: str = "available_measurement_count"):
        self.player_col = player_col
        self.count_col = count_col

    def fit(self, X, y=None):
        frame = pd.DataFrame(X, columns=[self.player_col, self.count_col]).copy()
        self.encoder_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.encoder_.fit(frame[[self.player_col]].astype("object"))
        self.feature_names_out_ = [
            f"{self.count_col}_x_{name}"
            for name in self.encoder_.get_feature_names_out([self.player_col])
        ]
        return self

    def transform(self, X):
        frame = pd.DataFrame(X, columns=[self.player_col, self.count_col]).copy()
        encoded = self.encoder_.transform(frame[[self.player_col]].astype("object"))
        counts = frame[self.count_col].astype(float).to_numpy().reshape(-1, 1)
        return encoded * counts

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_out_, dtype=object)


def make_f4_pipeline() -> Pipeline:
    feature_columns = F2_FEATURES.copy()
    forbidden = sorted(FORBIDDEN_FEATURES.intersection(feature_columns))
    if forbidden:
        raise RuntimeError(f"Forbidden feature in F4 base matrix: {forbidden}")
    if "available_measurement_count" not in feature_columns or "Player_Type" not in feature_columns:
        raise RuntimeError("F4 requires Player_Type and available_measurement_count.")

    categorical_cols = [column for column in CATEGORICAL_FEATURES if column in feature_columns]
    numeric_cols = [column for column in feature_columns if column not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median"), numeric_cols),
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_cols,
            ),
            (
                "availability_x_player_type",
                PlayerAvailabilityInteraction("Player_Type", "available_measurement_count"),
                INTERACTION_SOURCE_COLUMNS,
            ),
        ],
        remainder="drop",
    )
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=PROJECT_SEED, n_jobs=-1)
    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])


def positive_class_proba(fitted_pipeline: Pipeline, X_valid: pd.DataFrame) -> np.ndarray:
    model = fitted_pipeline.named_steps["model"]
    classes = np.asarray(model.classes_)
    matches = np.where(classes == 1)[0]
    if len(matches) != 1:
        raise RuntimeError(f"Expected class label 1 exactly once, got classes_={classes.tolist()}")
    proba = fitted_pipeline.predict_proba(X_valid)[:, int(matches[0])]
    if not np.isfinite(proba).all() or np.isnan(proba).any() or (proba < 0).any() or (proba > 1).any():
        raise RuntimeError("Predicted probabilities invalid.")
    return proba


def compute_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    if len(np.unique(y_true)) != 2:
        raise RuntimeError("ROC-AUC requested on one-class labels.")
    return float(roc_auc_score(y_true, y_score))


def train_f4() -> tuple[pd.DataFrame, list[dict]]:
    X = train_fe[F2_FEATURES].copy()
    oof = np.full(len(train_fe), np.nan, dtype=float)
    fold_rows = []
    for fold_id in range(5):
        valid_mask = fold_array == fold_id
        train_mask = ~valid_mask
        if len(np.unique(y[train_mask])) != 2 or len(np.unique(y[valid_mask])) != 2:
            raise RuntimeError(f"Fold {fold_id} has one-class train or validation target.")
        pipeline = make_f4_pipeline()
        pipeline.fit(X.loc[train_mask], y[train_mask])
        valid_proba = positive_class_proba(pipeline, X.loc[valid_mask])
        oof[valid_mask] = valid_proba
        fold_rows.append(
            {
                "fold": fold_id,
                "n_valid": int(valid_mask.sum()),
                "n_positive": int(y[valid_mask].sum()),
                "n_negative": int(valid_mask.sum() - y[valid_mask].sum()),
                "positive_rate": float(y[valid_mask].mean()),
                "f4_auc": compute_auc(y[valid_mask], valid_proba),
                "f2_auc": compute_auc(y[valid_mask], f2_proba[valid_mask]),
            }
        )
    if np.isnan(oof).any():
        raise RuntimeError("F4 OOF predictions contain NaN.")
    oof_df = pd.DataFrame({"Id": train["Id"], "fold": fold_array, "y_true": y, "y_pred_proba": oof})
    return oof_df, fold_rows

## 6. Execute Exactly One Variant

**Purpose.** Train only `phase7b_f4_role_count_player_type` and compare it
against persisted F2 OOF.

**Expected output.** In-memory F4 OOF predictions, fold metrics, and decision
statistics.

In [6]:
f4_oof, fold_rows = train_f4()
if list(f4_oof.columns) != ["Id", "fold", "y_true", "y_pred_proba"]:
    raise RuntimeError("F4 OOF columns invalid.")
if len(f4_oof) != 2781 or not f4_oof["Id"].equals(train["Id"]) or not f4_oof["fold"].equals(folds["fold"]):
    raise RuntimeError("F4 OOF row/order/fold alignment invalid.")
if not np.array_equal(f4_oof["y_true"].to_numpy(), y):
    raise RuntimeError("F4 y_true mismatch.")
if not np.isfinite(f4_oof["y_pred_proba"]).all() or not f4_oof["y_pred_proba"].between(0, 1).all():
    raise RuntimeError("F4 OOF probabilities invalid.")

f4_oof_auc = compute_auc(y, f4_oof["y_pred_proba"].to_numpy())
oof_delta = f4_oof_auc - f2_oof_auc
fold_detail = pd.DataFrame(fold_rows)
fold_detail["delta_f4_minus_f2"] = fold_detail["f4_auc"] - fold_detail["f2_auc"]
fold_detail["sign"] = np.where(fold_detail["delta_f4_minus_f2"] > 0, "positive", "non_positive")
same_sign_folds = int((fold_detail["delta_f4_minus_f2"] > 0).sum())

{
    "f2_oof_auc": f2_oof_auc,
    "f4_oof_auc": f4_oof_auc,
    "oof_delta_f4_minus_f2": oof_delta,
    "same_sign_folds": same_sign_folds,
}

{'f2_oof_auc': 0.8116502602456482,
 'f4_oof_auc': 0.809369070181826,
 'oof_delta_f4_minus_f2': -0.002281190063822214,
 'same_sign_folds': 1}

## 7. Slice Diagnostics

**Purpose.** Compare F4 vs F2 on mandatory slices. School appears only as a
diagnostic slice and never as a feature.

**Expected output.** Slice report and slice guard status.

In [7]:
SLICE_COLUMNS = [
    "Player_Type",
    "Position_Type",
    "Year",
    "Age_missing",
    "available_measurement_count",
    "measurement_completeness_group",
    "frequent_vs_rare_school_group",
]


def sorted_slice_values(series: pd.Series) -> list:
    values = series.dropna().unique().tolist()
    try:
        return sorted(values)
    except TypeError:
        return sorted(values, key=lambda value: str(value))


slice_rows = []
for slice_name in SLICE_COLUMNS:
    if slice_name not in train_fe.columns:
        slice_rows.append(
            {
                "slice_name": slice_name,
                "slice_value": "not_applicable",
                "n": 0,
                "n_positive": 0,
                "n_negative": 0,
                "positive_rate": np.nan,
                "f2_auc": np.nan,
                "f4_auc": np.nan,
                "delta_f4_minus_f2": np.nan,
                "status": "not_applicable",
                "reason_if_skipped": "slice column missing",
                "notes": "",
            }
        )
        continue
    for value in sorted_slice_values(train_fe[slice_name]):
        mask = train_fe[slice_name].eq(value).to_numpy()
        n = int(mask.sum())
        y_slice = y[mask]
        n_positive = int(y_slice.sum())
        n_negative = int(n - n_positive)
        positive_rate = float(y_slice.mean()) if n else np.nan
        status = "computed"
        reason = ""
        f2_auc_slice = np.nan
        f4_auc_slice = np.nan
        delta = np.nan
        if n < MIN_SLICE_SIZE:
            status = "skipped_too_small"
            reason = f"n<{MIN_SLICE_SIZE}"
        elif len(np.unique(y_slice)) < 2:
            status = "skipped_one_class"
            reason = "one target class"
        else:
            f2_auc_slice = compute_auc(y_slice, f2_proba[mask])
            f4_auc_slice = compute_auc(y_slice, f4_oof.loc[mask, "y_pred_proba"].to_numpy())
            delta = f4_auc_slice - f2_auc_slice
        slice_rows.append(
            {
                "slice_name": slice_name,
                "slice_value": str(value),
                "n": n,
                "n_positive": n_positive,
                "n_negative": n_negative,
                "positive_rate": positive_rate,
                "f2_auc": f2_auc_slice,
                "f4_auc": f4_auc_slice,
                "delta_f4_minus_f2": delta,
                "status": status,
                "reason_if_skipped": reason,
                "notes": "School grouping is diagnostic-only" if slice_name == "frequent_vs_rare_school_group" else "",
            }
        )

slice_report = pd.DataFrame(slice_rows)
computed_eval = slice_report[(slice_report["status"] == "computed") & (slice_report["n"] >= MIN_SLICE_SIZE)]
slice_guard_triggered = bool((computed_eval["delta_f4_minus_f2"].dropna() < -SLICE_DEGRADATION_THRESHOLD).any())
slice_guard_rows = computed_eval[computed_eval["delta_f4_minus_f2"] < -SLICE_DEGRADATION_THRESHOLD].copy()
slice_guard_triggered, slice_guard_rows[["slice_name", "slice_value", "n", "delta_f4_minus_f2"]].to_dict(orient="records")

(False, [])

## 8. Decision Classification and Artifact Writes

**Purpose.** Apply the F4 rule and write only authorized artifacts with
fail-if-exists guards.

**Expected output.** OOF, summary, slice report, validation report, candidate log.

In [8]:
if slice_guard_triggered:
    classification = "escalated"
    decision_reason = "Global result may pass or fail, but at least one mandatory slice degraded by more than 0.02 AUC."
elif oof_delta >= OOF_GAIN_THRESHOLD and same_sign_folds >= SAME_SIGN_FOLD_THRESHOLD:
    classification = "adopted"
    decision_reason = "OOF gain, same-sign fold criterion, and slice guard all passed."
else:
    classification = "rejected"
    decision_reason = "OOF gain and/or same-sign fold criterion did not pass."

variant_summary = pd.DataFrame(
    [
        {
            "experiment_id": EXPERIMENT_ID,
            "run_id": RUN_ID,
            "variant_id": VARIANT_ID,
            "baseline_variant_id": "phase7_f2_median_flags_count",
            "classification": classification,
            "decision_reason": decision_reason,
            "f2_oof_auc": f2_oof_auc,
            "f4_oof_auc": f4_oof_auc,
            "delta_f4_minus_f2_oof": oof_delta,
            "same_sign_positive_folds": same_sign_folds,
            "slice_guard_triggered": slice_guard_triggered,
            "fold_f2_auc_scores": "|".join(f"{v:.12f}" for v in fold_detail["f2_auc"]),
            "fold_f4_auc_scores": "|".join(f"{v:.12f}" for v in fold_detail["f4_auc"]),
            "fold_deltas_f4_minus_f2": "|".join(f"{v:.12f}" for v in fold_detail["delta_f4_minus_f2"]),
            "feature_set": "F2 + available_measurement_count_x_Player_Type",
            "model": "RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42,n_jobs=-1)",
            "leakage_warning": False,
            "hpo_run": False,
            "submission_created": False,
        }
    ]
)

def write_csv_fail_closed(df: pd.DataFrame, path: Path, label: str) -> None:
    assert_not_exists(path, label)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    view = df[columns].copy()
    for col in view.columns:
        if pd.api.types.is_float_dtype(view[col]):
            view[col] = view[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = ["| " + " | ".join(str(v) for v in row) + " |" for row in view.to_numpy()]
    return "\n".join([header, sep, *rows])


write_csv_fail_closed(f4_oof, OOF_PATH, "F4 OOF")
write_csv_fail_closed(variant_summary, VARIANT_SUMMARY_PATH, "variant summary")
write_csv_fail_closed(slice_report, SLICE_REPORT_PATH, "slice report")

candidate_log = pd.DataFrame(
    [
        {
            "experiment_id": EXPERIMENT_ID,
            "run_id": RUN_ID,
            "date": date.today().isoformat(),
            "phase": "Phase 7B - F4 role-aware measurement availability interaction probe",
            "notebook_or_script": "notebooks/07b_phase7b_role_availability_interaction_probe.ipynb",
            "git_commit_or_status": f"{head}; dirty_untracked_expected",
            "data_version": "official_data_input_train",
            "fold_strategy": "frozen_phase6_stratified_kfold_5_seed42",
            "random_seed": PROJECT_SEED,
            "feature_block": "F2 plus available_measurement_count_x_Player_Type",
            "preprocessing_summary": "Fold-safe median imputation, fold-fitted OHE for role categoricals, fold-fitted Player_Type OHE interaction.",
            "model_family": "RandomForestClassifier",
            "model_params_summary": "n_estimators=100; max_depth=5; random_state=42; n_jobs=-1",
            "hpo_status": "not_run",
            "f2_oof_auc": f2_oof_auc,
            "f4_oof_auc": f4_oof_auc,
            "delta_f4_minus_f2_oof": oof_delta,
            "same_sign_positive_folds": same_sign_folds,
            "slice_guard_triggered": slice_guard_triggered,
            "leakage_checks_passed": True,
            "submission_created": False,
            "submission_path": "",
            "public_lb_score_if_submitted": "",
            "decision": classification,
            "notes": decision_reason,
        }
    ]
)
write_csv_fail_closed(candidate_log, CANDIDATE_LOG_PATH, "candidate log")

top_slices = slice_report[(slice_report["status"] == "computed")].copy()
top_slices["abs_delta"] = top_slices["delta_f4_minus_f2"].abs()
top_slices = top_slices.sort_values(["abs_delta", "slice_name"], ascending=[False, True]).head(15)

report_lines = [
    f"# Phase 7B Validation Report - {EXPERIMENT_ID}",
    "",
    "## Executive Summary",
    "",
    f"F4 classification: `{classification}`.",
    f"F2 OOF ROC-AUC: `{f2_oof_auc:.12f}`.",
    f"F4 OOF ROC-AUC: `{f4_oof_auc:.12f}`.",
    f"OOF delta F4 - F2: `{oof_delta:.12f}`.",
    f"Same-sign fold count: `{same_sign_folds}/5`.",
    f"Slice guard triggered: `{slice_guard_triggered}`.",
    "",
    "## Authorized Hypothesis",
    "",
    "`available_measurement_count x Player_Type` is evaluated as the only F4 addition over F2. F4 is compared against persisted F2 OOF predictions, not against F0.",
    "",
    "## Environment and Contract",
    "",
    f"- Git HEAD: `{head}`",
    f"- Python: `{sys.version.split()[0]}`",
    f"- pandas: `{pd.__version__}`",
    f"- scikit-learn: `{sklearn.__version__}`",
    f"- numpy: `{np.__version__}`",
    f"- Frozen fold sha256[:16]: `{fold_hash}`",
    f"- Main experiment log unchanged: `pending final assertion`",
    "",
    "## F4 Implementation",
    "",
    "The interaction is implemented inside a scikit-learn transformer. `Player_Type` one-hot categories are fit only on each training fold, validation rows are transformed with `handle_unknown=\"ignore\"`, and the resulting one-hot columns are multiplied by the row-wise `available_measurement_count`.",
    "",
    "## Fold-Level Evidence",
    "",
    markdown_table(fold_detail, ["fold", "n_valid", "n_positive", "n_negative", "f2_auc", "f4_auc", "delta_f4_minus_f2", "sign"]),
    "",
    "## Slice Report Summary",
    "",
    markdown_table(top_slices, ["slice_name", "slice_value", "n", "f2_auc", "f4_auc", "delta_f4_minus_f2", "status"]),
    "",
    "## Leakage Checklist",
    "",
    "- No test fitting, tuning, preprocessing, feature selection, final inference, or submission occurred.",
    "- No global preprocessing was used for the F4 interaction.",
    "- `School` was not used as a feature; it was used only for the diagnostic frequent-vs-rare slice.",
    "- No external data, leaderboard data, HPO, model-family comparison, BMI, completeness bins as features, or extra interactions were used.",
    "- Positive-class probabilities were extracted only after verifying `estimator.classes_` contained label `1` exactly once.",
    "- Phase 8 remains locked.",
    "",
    "## Final Recommendation",
    "",
    decision_reason,
]
report_text = "\n".join(report_lines).replace("pending final assertion", "true") + "\n"
assert_not_exists(VALIDATION_REPORT_PATH, "validation report")
VALIDATION_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
VALIDATION_REPORT_PATH.write_text(report_text, encoding="utf-8")

main_log_after = read_bytes(MAIN_LOG_PATH)
if main_log_after != main_log_before:
    raise RuntimeError("logs/experiment_log.csv changed during Phase 7B execution.")

artifact_summary = {
    "oof_path": str(OOF_PATH.relative_to(PROJECT_ROOT)),
    "variant_summary_path": str(VARIANT_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
    "slice_report_path": str(SLICE_REPORT_PATH.relative_to(PROJECT_ROOT)),
    "validation_report_path": str(VALIDATION_REPORT_PATH.relative_to(PROJECT_ROOT)),
    "candidate_log_path": str(CANDIDATE_LOG_PATH.relative_to(PROJECT_ROOT)),
    "classification": classification,
    "main_log_unchanged": True,
}
artifact_summary

{'oof_path': 'outputs\\oof\\phase7b_role_availability_interaction_v1_phase7b_f4_role_count_player_type_oof_predictions.csv',
 'variant_summary_path': 'outputs\\validation\\phase7b_role_availability_interaction_v1_variant_summary.csv',
 'slice_report_path': 'outputs\\validation\\phase7b_role_availability_interaction_v1_slice_report.csv',
 'validation_report_path': 'outputs\\reports\\phase7b_role_availability_interaction_v1_validation_report.md',
 'candidate_log_path': 'outputs\\reports\\phase7b_role_availability_interaction_v1_experiment_log_candidate.csv',
 'classification': 'rejected',
 'main_log_unchanged': True}

## 9. Final Notebook Status

Phase 7B notebook execution is complete when all cells pass. Acceptance, staging,
commit, and Phase 8 remain locked behind explicit user/project director
authorization.